# PPM3 membrane workflow

Build a periodic protein–membrane system, export its parameters, and minimize it with pyCHARMM.
The example uses the offline 1gzm helix fragment, residues 35–65.
It uses ACE/CT3 caps and explicitly selects neutral HSE65.
HSE65 is a test assumption, not a biological protonation assignment.

The default membrane is symmetric POPC with generated TIP3 water and 0.15 M KCl.
Step 2 lists all 18 biological recipes and five PC/cholesterol families from OPRLM.
Select a preset there to use its composition throughout construction, visualization, and CHARMM minimization.

This workflow tests restrained preparation. It does not establish bilayer equilibration or readiness for unrestrained dynamics.

CRIMM uses PPM code obtained from Andrei Lomize and Irina Pogozheva, the main authors.
Keep PPM3 sources and local packages unpublished until distribution terms are recorded.
See the [PPM3 guide](../docs/membrane_ppm3.md) and [CHARMM testing guide](../docs/membrane_charmm_testing.md).

## 1. Select the environment

Open this notebook from the source checkout. Select a Python kernel with CRIMM, pyCHARMM, NGLView, RDKit, and NetworkX.
RDKit is required by the existing CRIMM visualization module.

If the pyCHARMM environment has no Jupyter kernel, install and register one from that environment:

```bash
python -m pip install ipykernel
python -m ipykernel install --user --name crimm --display-name "CRIMM / pyCHARMM"
```

Then select `CRIMM / pyCHARMM` in Jupyter.

Before you start Jupyter, set the CHARMM library directory:

```bash
export CHARMM_LIB_DIR=/path/to/charmm/lib
```

The directory must contain `libchmm.so` and its runtime dependencies.
If the native PPM3 library is absent, build it before you start the kernel:

```bash
python setup.py build_ext --inplace
```

A source build requires `gfortran`. There is no Python orientation fallback.

Run the cells in order. For another build, restart the kernel and run all cells.
Each run creates a new temporary output directory. Existing results remain unchanged.
NGLView requires a live Jupyter widget frontend. A static notebook preview does not show the interactive structures.

In [ ]:
from dataclasses import asdict
from importlib.util import find_spec
import io
import json
import os
from pathlib import Path
import subprocess
import sys
import tarfile
import tempfile

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / 'scripts/membrane_pycharmm.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook from the CRIMM source checkout.')
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
import crimm
from crimm.IO import PDBParser, write_crd, write_psf
from crimm.IO.PSFWriter import validate_psf
from crimm.Modeller import MembraneOrienter, TopologyGenerator
from crimm.Modeller.MembraneBuilder import MembraneBuilder, MembraneSpec
from crimm.Modeller.MembranePresets import get_membrane_preset, list_membrane_presets
from crimm.Modeller.Solvator import PERIODIC_WATER_OO_CUTOFF
from crimm.StructEntities.OrganizedModel import OrganizedModel
from crimm.Visualization import show_nglview_multiple
from scripts import membrane_pycharmm as workflow

assert Path(crimm.__file__).resolve().parent == ROOT / 'crimm', 'Restart the kernel to load this checkout.'
native_spec = find_spec('crimm.Modeller._ppm3._native')
if native_spec is None:
    raise RuntimeError('Build the native PPM3 library before you start this kernel.')
CHARMM_LIB = Path(os.environ.get('CHARMM_LIB_DIR', '/path/to/charmm/lib')).expanduser().resolve()
if not (CHARMM_LIB / 'libchmm.so').is_file():
    raise FileNotFoundError('Set CHARMM_LIB_DIR to the directory that contains libchmm.so.')
if find_spec('pycharmm') is None:
    raise ImportError('Select a kernel with pyCHARMM installed.')
OUTPUT = Path(tempfile.mkdtemp(prefix='crimm-ppm3-notebook-'))
print('Python:', sys.executable)
print('CRIMM:', crimm.__file__)
print('Output directory:', OUTPUT)

## 2. Set the membrane and protein direction

Positive Z is the upper, outer or luminal leaflet. Negative Z is the lower, inner or cytosolic leaflet.
`topology='in'` places the first available backbone N at negative Z. `topology='out'` places it at positive Z.
For this fragment, the reference N belongs to residue 35, not the biological protein terminus.

`lipid_z` sets the lipid-head planes. `membrane_core_z` sets the half-width of the water-exclusion slab.
These settings are separate from the native PPM3 hydrophobic thickness.

The preset selects explicit leaflet recipes, reference areas, and the matching PPM3 code.
The native positioning profile remains symmetric DOPC, not a composition-specific profile.

Run the catalog cell below, then edit the selection cell. Preset IDs are case-sensitive.
Use these settings for common examples:

| Membrane | `membrane_name` | `cholesterol_percent` |
| --- | --- | --- |
| Mammalian ER | `'ERm'` | `None` |
| Mammalian plasma | `'PMm'` | `None` |
| Plant plasma | `'PMp'` | `None` |
| Fungal plasma | `'PMf'` | `None` |
| POPC with 20 mol% cholesterol | `'POPC'` | `20.0` |

Biological presets use their reference temperature and box. Small boxes can lose minor components through rounding.
For the five PC families, set `pc_temperature_K` to match the intended physical phase.
The cholesterol value is a molecular percentage, not a mass percentage. Biological presets reject cholesterol overrides.
The selected composition table shows the requested percentage in each leaflet before whole-molecule rounding.

The aPM recipe remains experimental at reference density. Its strict packing test does not pass.
For the separate expanded aPM trial, select `'aPM'` with `cholesterol_percent = None`,
`initial_area_scale = 1.05`, and `short_range_warmup = True`.
This adds 5% to XY area after nominal lipid counts are fixed. It does not add lipids or scale Z.
The builder uses the expanded box for water generation, views, geometry checks, and CHARMM.
Expansion and short-range warmup remain explicit options. This notebook runs restrained minimization, without dynamics.
See the [preset guide](../docs/membrane_presets.md) for all IDs, source details, and limits.

For a custom asymmetric mixture, replace `MembraneSpec.from_preset(...)` with:

```python
spec = MembraneSpec(box_xy=cell[:2], box_z=cell[2],
    upper_lipid_ratios={'POPC': 40, 'DOPC': 40, 'CHL1': 20},
    lower_lipid_ratios={'DMPC': 25, 'DPPC': 25, 'DSPC': 30, 'CHL1': 20},
    random_seed=seed, max_repack_attempts=200)
```

Ratios are molecular proportions, including cholesterol. The builder normalizes them and rounds to whole molecules.
The default library has six types. Biological presets load 105 complete reference lipid types, including connected glycolipids and LPS.

In [ ]:
pc_families = {'POPC', 'DOPC', 'DSPC', 'DMPC', 'DPPC'}
catalogue = [get_membrane_preset(name, 0.0 if name in pc_families else None)
             for name in list_membrane_presets()]
preset_table = pd.DataFrame([
    {'ID': entry['id'], 'Membrane': entry['name'],
     'Temperature (K)': entry['temperature_K'] if entry['temperature_K'] is not None else 'Set below',
     'PPM3 code': entry['ppm_membrane']}
    for entry in catalogue
]).set_index('ID')
display(preset_table)

In [ ]:
membrane_name = 'POPC'
cholesterol_percent = 0.0  # Use None for a biological preset.
pc_temperature_K = 293.0  # Used only for PC/cholesterol families.
initial_area_scale = 1.0  # Use 1.05 for the expanded aPM trial.
short_range_warmup = False  # Use True for that aPM trial.
topology = 'in'
seed = 12345

In [ ]:
preset = get_membrane_preset(membrane_name, cholesterol_percent)
ppm_membrane = preset['ppm_membrane']
temperature_K = preset['temperature_K'] if preset['temperature_K'] is not None else pc_temperature_K
nominal_cell = tuple(preset.get('reference_box_A', (40.0, 40.0, 80.0)))
cell = nominal_cell
print('Selected:', preset['name'], 'Temperature (K):', temperature_K)
print('PPM3 code:', ppm_membrane, 'Positioning profile: DOPC')
print('Nominal box (A):', nominal_cell, 'Initial area scale:', initial_area_scale)
requested_percent = pd.DataFrame({
    'Outer (+Z) mol%': pd.Series(preset['upper_ratios'], dtype=float),
    'Inner (-Z) mol%': pd.Series(preset['lower_ratios'], dtype=float),
}).fillna(0.0)
requested_percent = requested_percent.div(requested_percent.sum()) * 100
display(requested_percent.round(2))

spec = MembraneSpec.from_preset(
    membrane_name, cholesterol_percent=cholesterol_percent,
    box_xy=cell[:2], box_z=cell[2],
    initial_area_scale=initial_area_scale,
    area_per_lipid=None, lipid_z=20.0, membrane_core_z=15.0,
    remove_membrane_core_waters=True, salt_concentration=0.15,
    cation='POT', anion='CLA', random_seed=seed, max_repack_attempts=200,
)
np.random.seed(seed)

## 3. Load the offline protein fragment

Load chain A and retain residues 35–65. No structure download is required.
The archive name records the older fixture source. Orientation still uses PPM3.

Select HSE65 explicitly before topology generation. The native hook rejects unresolved HIS, HSD, and HSP.
For another protein, determine the histidine states first. Do not change every histidine to HSE automatically.

In [ ]:
with tarfile.open(ROOT / 'tests/data/ppm2_reference.tar.gz') as archive:
    member = next(item for item in archive.getmembers() if Path(item.name).name == '1gzm.pdb')
    pdb_text = archive.extractfile(member).read().decode('ascii')
model = PDBParser(QUIET=True).get_structure(io.StringIO(pdb_text), structure_id='1gzm')[0]
for chain in list(model):
    if chain.id != 'A' or chain.chain_type != 'Polypeptide(L)':
        model.detach_child(chain.id)
        continue
    for residue in list(chain):
        if not 35 <= residue.id[1] <= 65:
            chain.detach_child(residue.id)
residues = list(model.get_residues())
assert len(residues) == 31
histidine = next(residue for residue in residues if residue.id[1] == 65)
assert histidine.resname == 'HIS', 'The reference fixture changed.'
histidine.resname = 'HSE'
model = OrganizedModel(model, fetch_web_data=False)
print('Protein residues:', len(list(model.get_residues())))
print('Histidine choice:', histidine.resname, histidine.id[1])

## 4. Build the protein topology and missing atoms

Apply ACE and CT3 caps to this isolated fragment. Build missing coordinates and hydrogens before the native orientation call.
For a complete protein, select the terminal patches that match its chemistry.

In [ ]:
topo = TopologyGenerator()
topo.generate_model(model, prot_first_patch='ACE', prot_last_patch='CT3',
                    build_coords=True, QUIET=True)
assert all(not residue.missing_atoms and not residue.missing_hydrogens
           for residue in model.get_residues())
print('Complete protein atoms:', len(list(model.get_atoms())))

## 5. Orient the protein with native PPM3

Apply the native rigid transform to the complete model. Check the rotation and the requested N direction.
The reported tilt is the native optimizer angle, not a fitted helix-axis angle.

In [ ]:
before_orientation = np.array([atom.coord.copy() for atom in model.get_atoms()])
orientation = MembraneOrienter(
    model, ph=7.5, temperature=temperature_K, membrane_type=ppm_membrane, topology=topology,
).orient(in_place=True)
np.testing.assert_allclose(np.linalg.det(orientation.rotation), 1.0, atol=2e-5, rtol=0)
np.testing.assert_allclose(
    [atom.coord for atom in model.get_atoms()],
    before_orientation @ orientation.rotation.T + orientation.translation, atol=1e-8, rtol=0,
)
assert orientation.topology_reference['final_z'] * (1 if topology == 'out' else -1) > 0.002
orientation_record = asdict(orientation)
orientation_record['rotation'] = orientation.rotation.tolist()
orientation_record['translation'] = orientation.translation.tolist()
display(pd.Series({
    'Native version': orientation.native_version,
    'Native membrane code': orientation.membrane_type,
    'Positioning profile': orientation.profile,
    'Hydrophobic thickness (A)': orientation.membrane_thickness,
    'Native tilt (degrees)': orientation.tilt,
    'Reference N Z (A)': orientation.topology_reference['final_z'],
    'Topology': orientation.topology,
}, name='PPM3 result'))

### View the orientation

Drag to rotate the structure. Scroll to zoom.
Purple shows the protein. Grey frames mark the nominal lipid-head planes at Z = ±20 Å.
Orange frames mark the excluded membrane core at Z = ±15 Å.
If you select area expansion, the expanded XY frames appear after Step 6.
The frames are visual guides, not atoms or restraints.
Later views show upper heads in green, lower heads in red, water in blue, and ions in magenta.

In [ ]:
def system_view(entity, show_water=False):
    chains = [chain for chain in entity
              if show_water or chain.chain_type not in {'Solvent', 'Ion'}]
    view = show_nglview_multiple(chains)
    view.clear_representations()
    for index, chain in enumerate(chains):
        if chain.chain_type == 'Polypeptide(L)':
            view.add_cartoon(component=index, color='purple')
        elif chain.chain_type in {'Lipid', 'Sterol'}:
            view.add_licorice(component=index, selection='not hydrogen', color='grey', opacity=0.5)
            head_ids = {id(atom) for residue in chain
                        for atom in MembraneBuilder._head_atoms(residue, chain.chain_type)}
            for side, color in ((1, 'green'), (-1, 'red')):
                heads = [i for i, atom in enumerate(chain.get_atoms())
                         if id(atom) in head_ids and side * atom.coord[2] > 0]
                if heads:
                    view.add_spacefill(component=index, selection=heads, color=color, radius=0.9)
        elif chain.chain_type == 'Solvent':
            oxygens = [i for i, atom in enumerate(chain.get_atoms()) if atom.element == 'O']
            view.add_point(component=index, selection=oxygens, color='blue', point_size=2)
        elif chain.chain_type == 'Ion':
            view.add_spacefill(component=index, color='magenta', radius=0.8)
    x, y = spec.box_xy[0] / 2, spec.box_xy[1] / 2
    for height, color in ((spec.lipid_z, [0.5, 0.5, 0.5]),
                          (spec.membrane_core_z, [1.0, 0.5, 0.0])):
        for z in (-height, height):
            corners = [[-x, -y, z], [x, -y, z], [x, y, z], [-x, y, z]]
            for first, last in zip(corners, corners[1:] + corners[:1]):
                view.shape.add_cylinder(first, last, color, 0.12)
    view.camera = 'orthographic'
    view.background = 'white'
    view.layout.width = '100%'
    view.layout.height = '500px'
    view.center()
    return view

oriented_view = system_view(model)
display(oriented_view)

## 6. Build the lipid bilayer

Reuse the protein topology loader for the membrane. The original six-type library retains 2,000 conformers per type.
Biological presets use all whole lipid molecules from the cited reference archives in a separate compact library.
The builder loads the library once. It does not load archived water positions.
Check the requested and realized composition after whole-molecule rounding.

In [ ]:
builder = MembraneBuilder(model, spec)
spec = builder.spec
cell = (*spec.box_xy, spec.box_z)
print('Build box (A):', cell)
assert builder.topology is model.topology_loader
builder.prepare_topology()
builder.add_lipid_bilayer()
display(pd.DataFrame({'upper (+Z)': builder.upper_leaflet,
                      'lower (-Z)': builder.lower_leaflet}).fillna(0).astype(int))
print(json.dumps(builder.leaflet_composition, indent=2))
bilayer_view = system_view(model)
display(bilayer_view)

## 7. Generate water outside the membrane core

Generate periodic TIP3 water with the existing Solvator. Keep the native orientation unchanged.
If any water atom enters `abs(z) < membrane_core_z`, remove that complete water molecule.
Check all three atoms of every retained water, not only its oxygen.

In [ ]:
builder.solvate()
waters = [residue for residue in model.get_residues() if residue.resname == 'TIP3']
assert waters
assert all(set(residue.child_dict) == {'OH2', 'H1', 'H2'} for residue in waters)
assert all(abs(float(atom.coord[2])) >= spec.membrane_core_z
           for residue in waters for atom in residue)
print('Generated water molecules:', builder.count_waters())

## 8. Add salt and check the constructed system

Replace water molecules with POT and CLA ions. G-OM first adds five CAL ions per outer-leaflet LPS molecule.
Check the calculated ion counts and total charge. Calcium replaces generated water outside the positive-Z membrane core.
Reuse the existing periodic geometry checks for contacts, chirality, cis bonds, and ring penetration.
Initial lipid hydrogen contacts can remain. The CHARMM step relaxes hydrogens before whole-system minimization.

In [ ]:
builder.add_ions()
fixed_ions = model._solvation_info.get('fixed_ion_counts', {})
ion_counts = {name: sum(residue.resname == name for residue in model.get_residues())
              for name in ('POT', 'CLA', *fixed_ions)}
assert ion_counts == {'POT': model._solvation_info['n_cation'],
                      'CLA': model._solvation_info['n_anion'], **fixed_ions}
initial_geometry = workflow.geometry_checks(model, builder.lipid_library, cell, spec.membrane_core_z)
for key in ('reversed_lipids', 'trans_lipids', 'inward_lipid_heads',
            'ring_penetrations', 'waters_in_core', 'ions_in_core',
            'periodic_heavy_contacts_below_1_2_A'):
    assert not initial_geometry[key], f'{key}: {initial_geometry[key]}'
assert initial_geometry['minimum_contact_A'] != 0
assert max(initial_geometry['periodic_water_face_gaps_A']) <= 2.0
assert initial_geometry['minimum_water_oxygen_to_solute_A'] >= spec.water_solvcut - 1e-8
assert initial_geometry['minimum_water_oxygen_pair_A'] >= PERIODIC_WATER_OO_CUTOFF - 1e-8
assert initial_geometry['actual_leaflet_composition'] == [
    {name: count for name, count in leaf.items() if count}
    for leaf in (builder.upper_leaflet, builder.lower_leaflet)
]
display(pd.Series({'Lipids': builder.count_lipids(), 'Sterols': builder.count_sterols(),
                   'Water molecules': builder.count_waters(), **ion_counts}, name='Molecule counts'))
solvated_view = system_view(model, show_water=True)
display(solvated_view)

## 9. Export the complete CHARMM input

Export the PSF, CRD, topology, parameters, and loading stream. A PSF and CRD alone are not sufficient for minimization.
Check complete residues, PSF consistency, atom identities, and neutrality before CHARMM loading.
Record the input and source hashes for the fresh CHARMM process.

In [ ]:
assert all(not residue.missing_atoms and not residue.missing_hydrogens
           for residue in model.get_residues())
assert not validate_psf(model, strict=True)
write_psf(model, str(OUTPUT / 'system.psf'))
write_crd(model, str(OUTPUT / 'system.crd'))
toppar_files = topo.write_toppar(path=str(OUTPUT))
psf_data, _, _ = workflow.coordinate_pair(OUTPUT)
charge = sum(atom.charge for atom in psf_data.atoms)
assert abs(charge) < 1e-4
source_files = [
    'scripts/membrane_pycharmm.py', 'crimm/Modeller/MembraneBuilder.py',
    'crimm/Modeller/Solvator.py', 'crimm/Modeller/TopoLoader.py',
    'crimm/Modeller/MembraneOrienter.py', 'crimm/charmm_gui_9qsy/pentest.py',
    'crimm/Data/lipid_conformers.npz', 'crimm/IO/PSFWriter.py', 'crimm/IO/CRDWriter.py',
    'crimm/Modeller/_ppm3/res.lib',
    'crimm/Modeller/MembranePresets.py', 'crimm/Data/oprlm_lipids.json',
    'crimm/Data/oprlm_lipids.npz', 'crimm/charmm_gui_9qsy/checkfft.py',
]
input_files = [OUTPUT / 'system.psf', OUTPUT / 'system.crd', *map(Path, toppar_files)]
record = {
    'case': 'helix', 'seed': seed, 'dry': False, 'status': 'running',
    'python': sys.version, 'cell_A': cell, 'membrane_core_z_A': spec.membrane_core_z,
    'temperature_K': temperature_K,
    'protein_model': '1gzm residues 35-65, ACE/CT3, explicitly selected neutral HSE65',
    'protein_fixture_archive_sha256': workflow.digest(ROOT / 'tests/data/ppm2_reference.tar.gz'),
    'orientation': orientation_record,
    'membrane_preset': preset,
    'initial_preparation': {'nominal_cell_A': nominal_cell, 'area_scale': initial_area_scale,
                            'lipid_counts_fixed_before_expansion': True},
    'short_range_warmup': short_range_warmup,
    'lipid_geometry': {name: {key: template[key] for key in
        ('head_atoms', 'chiral_centres', 'dihedral_guards')}
        for name, template in builder.lipid_library.templates.items()
        if name in builder.upper_leaflet or name in builder.lower_leaflet},
    'protein_segments': sorted({residue.segid.strip() for chain in model
                                if chain.chain_type == 'Polypeptide(L)' for residue in chain}),
    'natom': len(psf_data.atoms), 'charge_e': charge, 'initial_geometry': initial_geometry,
    'leaflet_counts': [builder.upper_leaflet, builder.lower_leaflet],
    'leaflet_composition': builder.leaflet_composition, 'actual_ion_counts': ion_counts,
    'counts': {'lipids': builder.count_lipids(), 'sterols': builder.count_sterols(),
               'water': builder.count_waters(), 'ions': builder.count_ions()},
    'solvation': dict(model._solvation_info),
    'topology_counts': {name: len(getattr(psf_data, name))
                        for name in ('bonds', 'angles', 'dihedrals', 'impropers', 'cmap')},
    'input_sha256': {str(path.relative_to(OUTPUT)): workflow.digest(path) for path in input_files},
    'source_sha256': {name: workflow.digest(ROOT / name) for name in source_files},
    'ppm3_library_path': native_spec.origin, 'ppm3_library_sha256': workflow.digest(native_spec.origin),
    'charmm_library': str(CHARMM_LIB / 'libchmm.so'),
    'charmm_library_sha256': workflow.digest(CHARMM_LIB / 'libchmm.so'),
    'scope': 'single solvated helix preparation, not equilibration',
}
workflow.save_record(OUTPUT, record, 'charmm_start')
print('Atoms:', record['natom'], 'Charge (e):', charge)
print((OUTPUT / 'load_toppar.str').read_text())

## 10. Load and minimize with pyCHARMM

Run the existing [minimization worker](../scripts/membrane_pycharmm.py) in a fresh process. Use the same Python interpreter as the kernel.
The worker checks the exported parameters, atom inventory, charges, masses, and coordinates.
It defines the periodic crystal, images, and PME before energy evaluation.

The worker first relaxes hydrogens with heavy atoms fixed. It then releases fixed atoms and minimizes the complete system.
With `short_range_warmup = True`, hydrogen relaxation uses a 6 Å cutoff without PME.
The worker restores full PME settings before it releases heavy atoms and minimizes the complete system.
Harmonic forces decrease through 10, 1, and 0.1 kcal mol⁻¹ Å⁻².
Protein heavy-atom restraints act along XYZ. Lipid-head restraints act along Z.
Dihedral guards preserve lipid chirality and cis bonds. A soft water guard protects the membrane core during preparation.

The final restrained RMS gradient must be at most 1.0 kcal mol⁻¹ Å⁻¹.
The worker then clears restraints and records a separate unrestrained energy and gradient without further minimization.
Do not interpret restrained convergence as unrestrained convergence.

This step takes several minutes. The complete CHARMM output is saved in `charmm.log`.
If the process fails, read its recorded stage and that log before you continue.

In [ ]:
command = [sys.executable, str(ROOT / 'scripts/membrane_pycharmm.py'),
           '--worker', '--case', 'helix', '--output', str(OUTPUT),
           '--charmm-lib', str(CHARMM_LIB)]
env = dict(os.environ, CHARMM_LIB_DIR=str(CHARMM_LIB),
           NUMBA_CACHE_DIR=str(OUTPUT / 'numba-cache'), OMP_NUM_THREADS='1')
print('CHARMM log:', OUTPUT / 'charmm.log', flush=True)
with (OUTPUT / 'charmm.log').open('w') as log:
    completed = subprocess.run(command, cwd=OUTPUT, env=env, stdout=log,
                               stderr=subprocess.STDOUT,
                               timeout=3600 if preset['kind'] == 'biological' else 900)
record = json.loads((OUTPUT / 'result.json').read_text())
if completed.returncode or record['stage'] != 'charmm_complete':
    raise RuntimeError(f"CHARMM failed at {record['stage']}. Read {OUTPUT / 'charmm.log'}.")
assert record['charmm_diagnostic_passed'], 'Read the energy and gradient checks in result.json.'
print('CHARMM loading and restrained minimization passed.')

## 11. Check and view the minimized system

Load coordinates by PSF/CRD atom identity, not by an unchecked list order.
Check all lipid atoms for periodic contacts. Check water and ions outside the membrane core again.
Check leaflet composition, chirality, cis bonds, ring penetration, and protein placement after minimization.

In [ ]:
workflow.save_record(OUTPUT, record, 'final_validation')
_, entries, minimized_coords = workflow.coordinate_pair(OUTPUT, 'minimized.crd')
atoms_by_identity = {(residue.segid.strip(), str(residue.id[1]), residue.resname, atom.name): atom
                     for residue in model.get_residues() for atom in residue}
identities = [(entry.segid, str(entry.resid), entry.resname, entry.atomname) for entry in entries]
assert len(atoms_by_identity) == len(identities) == len(set(identities))
assert set(identities) == set(atoms_by_identity)
for identity, coord in zip(identities, minimized_coords):
    atoms_by_identity[identity].coord = coord
final_geometry = workflow.geometry_checks(model, builder.lipid_library, cell, spec.membrane_core_z)
for key in ('reversed_lipids', 'trans_lipids', 'inward_lipid_heads', 'ring_penetrations',
            'waters_in_core', 'ions_in_core', 'periodic_contacts_below_1_2_A'):
    assert not final_geometry[key], f'{key}: {final_geometry[key]}'
assert final_geometry['actual_leaflet_composition'] == initial_geometry['actual_leaflet_composition']
assert max(final_geometry['periodic_water_face_gaps_A']) <= 2.0
protein_geometry = workflow.protein_checks(OUTPUT, record['protein_segments'], record['orientation'])
assert protein_geometry['passed'], protein_geometry
record.update(final_geometry=final_geometry, protein_geometry=protein_geometry,
              construction_checks_passed=True, case_acceptance_passed=True, status='passed',
              full_membrane_acceptance_matrix='not run')
workflow.save_record(OUTPUT, record, 'complete')
display(pd.DataFrame({'Initial': initial_geometry, 'Minimized': final_geometry}).loc[
    ['waters_in_core', 'ions_in_core', 'periodic_contacts_below_1_2_A',
     'periodic_heavy_contacts_below_1_2_A', 'actual_leaflet_composition']
])
display(pd.Series(protein_geometry, name='Protein checks'))

In [ ]:
display(pd.DataFrame([
    {'Harmonic force': stage['force'], 'Energy (kcal/mol)': stage['energy']['ENER'],
     'RMS gradient (kcal/mol/A)': stage['energy']['GRMS']}
    for stage in record['stages']
]))
display(pd.DataFrame({
    label: {'Energy (kcal/mol)': record[key]['ENER'],
            'RMS gradient (kcal/mol/A)': record[key]['GRMS']}
    for label, key in (('Initial, final restraints', 'initial_final_hamiltonian'),
                       ('Minimized, final restraints', 'final_final_hamiltonian'),
                       ('Minimized, restraints cleared', 'final_unrestrained'))
}).T)
assert record['same_hamiltonian_energy_decreased']
assert record['final_final_hamiltonian']['GRMS'] <= record['target_grms']
minimized_view = system_view(model, show_water=True)
display(minimized_view)

## 12. Keep the results

Copy the output directory to permanent storage before temporary-file cleanup.
`system.crd` contains the initial coordinates. `minimized.crd` contains the prepared coordinates.
`toppar/` and `load_toppar.str` contain the required topology and parameters.
`result.json` contains the checks and provenance. `charmm.log` contains the complete CHARMM output.

Compare energy changes only with the same restraints and Hamiltonian.
The final unrestrained snapshot is a diagnostic, not a convergence test.
Before production dynamics, define and test a separate membrane equilibration protocol.
For the full preparation matrix, use [the existing matrix runner](../scripts/membrane_matrix.py).

In [ ]:
print('Saved results:', OUTPUT)
display(pd.DataFrame([
    {'File': str(path.relative_to(OUTPUT)), 'Bytes': path.stat().st_size}
    for path in sorted(OUTPUT.rglob('*'))
    if path.is_file() and 'numba-cache' not in path.parts
]))